**Install Libraries**

In [2]:
#!pip install transformers datasets trl==0.7.2 accelerate peft -q

**Load Model and Tokenizer**

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


**Load a Clean Instruction Dataset**

In [4]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train[:1%]")

def format_data(example):
    text = "Instruction: " + example["instruction"] + "\nAnswer: " + example["output"]
    return {"text": text}

dataset = dataset.map(format_data)

print(dataset[0])

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.', 'text': 'Instruction: Give three tips for staying healthy.\nAnswer: 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of f

**Tokenize Dataset**

In [5]:
max_length = 128

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

print(tokenized_dataset[0])

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.', 'text': 'Instruction: Give three tips for staying healthy.\nAnswer: 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of f

**Supervised Fine-Tuning (SFT)**

In [6]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./sft_model",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.067795
20,2.199081
30,2.122384
40,1.864326
50,2.171404
60,1.967181
70,1.792476
80,1.727925
90,1.998754
100,1.610400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=259, training_loss=1.9400927679879325, metrics={'train_runtime': 21.5153, 'train_samples_per_second': 24.076, 'train_steps_per_second': 12.038, 'total_flos': 16918964600832.0, 'train_loss': 1.9400927679879325, 'epoch': 1.0})

**Create Safe Preference Dataset**

In [7]:
from datasets import Dataset

preference_data = [
    {
        "prompt": "Explain gravity simply.",
        "chosen": "Gravity is the force that pulls objects toward Earth.",
        "rejected": "Gravity is when things randomly float."
    },
    {
        "prompt": "What is photosynthesis?",
        "chosen": "Plants use sunlight to make food.",
        "rejected": "Plants eat soil to survive."
    }
]

pref_dataset = Dataset.from_list(preference_data)

print(pref_dataset[0])

{'prompt': 'Explain gravity simply.', 'chosen': 'Gravity is the force that pulls objects toward Earth.', 'rejected': 'Gravity is when things randomly float.'}


**Load Reward Model**

In [8]:
from transformers import AutoModelForSequenceClassification

reward_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
)

reward_tokenizer = AutoTokenizer.from_pretrained(model_name)
reward_tokenizer.pad_token = reward_tokenizer.eos_token

print("Reward model loaded")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key                                        | Status     | 
-------------------------------------------+------------+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED | 
score.weight                               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Reward model loaded


**Tokenize Preference Data**

In [9]:
def tokenize_pref(example):
    chosen = example["prompt"] + " " + example["chosen"]
    rejected = example["prompt"] + " " + example["rejected"]

    chosen_tokens = reward_tokenizer(chosen, truncation=True, padding="max_length", max_length=128)
    rejected_tokens = reward_tokenizer(rejected, truncation=True, padding="max_length", max_length=128)

    return {
        "input_ids_chosen": chosen_tokens["input_ids"],
        "attention_mask_chosen": chosen_tokens["attention_mask"],
        "input_ids_rejected": rejected_tokens["input_ids"],
        "attention_mask_rejected": rejected_tokens["attention_mask"],
    }

tokenized_pref = pref_dataset.map(tokenize_pref)

print(tokenized_pref[0])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

{'prompt': 'Explain gravity simply.', 'chosen': 'Gravity is the force that pulls objects toward Earth.', 'rejected': 'Gravity is when things randomly float.', 'input_ids_chosen': [18438, 391, 13522, 2391, 13, 24084, 318, 262, 2700, 326, 16194, 5563, 3812, 3668, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,

**Train Reward Model**

In [10]:
!mkdir -p /usr/local/lib/python3.12/dist-packages/trl/templates/
!touch /usr/local/lib/python3.12/dist-packages/trl/templates/rm_model_card.md

In [11]:
#!pip install -U trl


In [12]:
from trl import RewardTrainer, RewardConfig

reward_args = RewardConfig(
    output_dir="./reward_model",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    report_to="none"
)

reward_trainer = RewardTrainer(
    model=reward_model,
    args=reward_args,
    train_dataset=tokenized_pref
)

reward_trainer.train()
print("Reward model training completed.")

Adding EOS to train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Filtering train >1024 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

Step,Training Loss
1,0.321763
2,1.074166


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Reward model training completed.


**RLHF via PPO**

In [13]:
!mkdir -p /usr/local/lib/python3.12/dist-packages/trl/templates/
!touch /usr/local/lib/python3.12/dist-packages/trl/templates/lm_model_card.md

In [15]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl.experimental.ppo import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from transformers.modeling_outputs import CausalLMOutputWithPast

model_name = "distilgpt2"

# 1. Tokenizer Setup (PPO requires left-padding)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 2. Initialize Models
# Reward & Value models must be regression (num_labels=1)
reward_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
value_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

# Policy & Reference models (Wrapped)
model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)

# 3. FIX: Untie weights to prevent Safetensors/Shared Memory errors
# CausalLM models use 'lm_head'; SequenceClassification models use 'score'
for m in [model, ref_model]:
    if hasattr(m.pretrained_model, "lm_head"):
        m.pretrained_model.lm_head.weight = torch.nn.Parameter(m.pretrained_model.lm_head.weight.clone())
    m.pretrained_model.config.tie_word_embeddings = False

for m in [reward_model, value_model]:
    if hasattr(m, "score"):
        m.score.weight = torch.nn.Parameter(m.score.weight.clone())
    m.config.tie_word_embeddings = False

# 4. FIX: Handle the 'AttributeError: tuple object has no attribute logits'
def forward_wrapper(original_forward):
    def wrapped(*args, **kwargs):
        kwargs["return_dict"] = True
        out = original_forward(*args, **kwargs)
        if isinstance(out, tuple):
            return CausalLMOutputWithPast(logits=out[0])
        return out
    return wrapped

model.forward = forward_wrapper(model.forward)
ref_model.forward = forward_wrapper(ref_model.forward)

# 5. FIX: Link missing attributes to the wrapper for the Trainer
for m in [model, ref_model]:
    m.generation_config = m.pretrained_model.generation_config
    m.is_gradient_checkpointing = getattr(m.pretrained_model, "is_gradient_checkpointing", False)
    m.config.return_dict = True

# 6. Dataset Cleaning
# Strip everything except 'input_ids' to avoid string-to-tensor errors
ppo_dataset = tokenized_dataset.select_columns(["input_ids"])

# 7. PPO Configuration
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=2,
    mini_batch_size=1,
    num_ppo_epochs=1,
    save_strategy="no" # Required for GPT-2 weight-tying issues
)

# 8. Initialize Trainer
ppo_trainer = PPOTrainer(
    ppo_config,
    processing_class=tokenizer,
    model=model,
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_dataset
)

# 9. Start Training
ppo_trainer.train()

print("RLHF via PPO training completed.")


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key                                        | Status     | 
-------------------------------------------+------------+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED | 
score.weight                               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key                                        | Status     | 
-------------------------------------------+------------+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED | 
score.weight                               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

...

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


===training policy===


/usr/local/lib/python3.12/dist-packages/trl/experimental/ppo/ppo_trainer.py:899: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  metrics["val/ratio_var"] = self.accelerator.gather_for_metrics(ratio_stats).var().item()


Step,Training Loss


RLHF via PPO training completed.


**Test the Fine-Tuned Model**

In [21]:
import torch

prompt = "Instruction: Explain what is gen AI.\nAnswer:"

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Instruction: Explain what is gen AI.
Answer: "Gen AI is the perfect tool for a human being."You're not a "doomsday" moment for the White House.


A federal judge has ruled that a federal judge in Texas is going to step down from the Obama administration's ban on immigration from Muslim-majority countries.
